# Data Cleaning Pipeline: Abnormal Listing Detection & Product Recommendation

Notebook นี้ทำความสะอาดและเตรียมข้อมูลให้พร้อมสำหรับโมเดล AI จาก 2 ชุดข้อมูล:

1. **`dataset_abnormal_listing.xlsx`** — AI Abnormal Listing Detection (ตรวจจับประกาศขายสินค้าผิดปกติ)
2. **`dataset_recommendation.xlsx`** — Product Recommendation System

**วิธีใช้:** วางไฟล์ทั้งสองไว้ในโฟลเดอร์ `data/` (ข้างๆ ไฟล์ notebook นี้) แล้วรันทีละเซลล์ตามลำดับ
หากไฟล์ของคุณชื่ออื่น ให้แก้ตัวแปร `FILE_1` และ `FILE_2` ในเซลล์ Setup ด้านล่าง

Output: ไฟล์ CSV ที่ทำความสะอาดแล้ว จะถูกเซฟไว้ในโฟลเดอร์ `output/`


## 0. Setup

In [9]:
import pandas as pd
import numpy as np
import re
import os

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

FILE_1 = "data/dataset_abnormal_listing.xlsx"   # Abnormal Listing Detection
FILE_2 = "data/dataset_recommendation.xlsx"     # Product Recommendation

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


---
# PART 1 — Abnormal Listing Detection Dataset

โครงสร้างไฟล์: `1_Data_Dictionary`, `2_Business_Rules`, `3_Training_Dataset`,
`4_Validation_Dataset`, `5_Testing_Dataset`, `6_UAT_Dataset`, `7_Coverage_Matrix`, `8_Acceptance_Criteria`

แต่ละชีทข้อมูล (Training/Validation/Testing/UAT) มี header อยู่ที่แถวที่ 5 ของ Excel (index 4)


In [10]:
xl1 = pd.ExcelFile(FILE_1)
print(xl1.sheet_names)


['1_Data_Dictionary', '2_Business_Rules', '3_Training_Dataset', '4_Validation_Dataset', '5_Testing_Dataset', '6_UAT_Dataset', '7_Coverage_Matrix', '8_Acceptance_Criteria']


In [11]:
# โหลด Business Rules และ Data Dictionary ไว้อ้างอิงระหว่างทำความสะอาด
bus_rules_1 = xl1.parse("2_Business_Rules", header=3)
data_dict_1 = xl1.parse("1_Data_Dictionary", header=3)
bus_rules_1


,Rule ID,Title,Description,Example Input,Expected Output,Severity,Applies To Scenario Types
0,BR001,Extreme Price Anomaly,Detect listings where price is grossly disprop...,"Pencil priced at 1,000,000 THB",REJECT — Abnormal Price,Critical,PRICE_ANOMALY
1,BR002,High Price Anomaly,Detect prices significantly higher than the ma...,"Marketplace average 500 THB, listing price 25,...",REVIEW — High Price Anomaly,High,"PRICE_ANOMALY, BOUNDARY"
2,BR003,Low Price / Scam Anomaly,Detect prices significantly lower than the mar...,iPhone 15 listed at 500 THB,REVIEW/REJECT — Possible Scam,Critical,"PRICE_ANOMALY, SCAM, FRAUD"
3,BR004,Prohibited Descriptions,"Detect descriptions referencing adult, unsanit...","""Used underwear, worn for 2 years""",REJECT — Adult/Unsanitary Product,Critical,PROHIBITED_PRODUCT
4,BR005,Misleading Description,Detect a mismatch between the product title an...,Title: iPhone 15 / Description: Android phone,REVIEW — Mismatch,High,DESCRIPTION_ANOMALY
5,BR006,Category Mismatch,Detect a mismatch between the declared categor...,Category: Laptop / Image: Shoes,REVIEW — Category Mismatch,Medium,CATEGORY_MISMATCH
6,BR007,Duplicate Listing,Detect the same seller re-posting an identical...,"Same seller, same image, different title",REVIEW — Spam Listing,Medium,DUPLICATE
7,BR008,Keyword Stuffing,Detect titles/descriptions stuffed with excess...,"""iPhone Apple Samsung Sony ASUS Lenovo Cheap C...",REVIEW — Spam Description,Medium,KEYWORD_SPAM
8,BR009,Fake Urgency Marketing,Detect exaggerated urgency marketing language ...,"""LAST ONE!!! BUY NOW!!! ONLY TODAY!!!""",REVIEW — Marketing Spam,Low,SPAM
9,BR010,Prohibited Goods,"Detect listings for weapons, drugs, adult prod...",Listing for an unregistered handgun,REJECT,Critical,"PROHIBITED_PRODUCT, COUNTERFEIT"


In [12]:
def load_listing_sheet(sheet_name):
    df = xl1.parse(sheet_name, header=4)
    df["__source_sheet__"] = sheet_name
    return df

train_raw = load_listing_sheet("3_Training_Dataset")
val_raw   = load_listing_sheet("4_Validation_Dataset")
test_raw  = load_listing_sheet("5_Testing_Dataset")
uat_raw   = load_listing_sheet("6_UAT_Dataset")

for name, df in [("train", train_raw), ("val", val_raw), ("test", test_raw), ("uat", uat_raw)]:
    print(name, df.shape)


train (20000, 45)
val (2000, 45)
test (2000, 45)
uat (1000, 45)


### 1.1 รวมชุดข้อมูลชั่วคราวเพื่อทำความสะอาดพร้อมกัน

รวมทั้ง 4 ชุดเข้าด้วยกันก่อน (มี `__source_sheet__` กำกับไว้) เพื่อให้ทำความสะอาดครั้งเดียว
แล้วค่อยแยกกลับตามชุดเดิมตอนท้าย — วิธีนี้การันตีว่ากติกาทำความสะอาดเหมือนกันทุกชุด


In [13]:
listing_all = pd.concat([train_raw, val_raw, test_raw, uat_raw], ignore_index=True)
print(listing_all.shape)
listing_all.head(3)


(25000, 45)


,Dataset ID,Scenario ID,Scenario Type,Listing ID,Seller ID,Category,Subcategory,Brand,Model,Product Name,Title,Description,Condition,Price,Marketplace Average Price,Price Deviation Percentage,Seller Rating,Seller Total Listings,Seller Violation Count,Duplicate Image,Duplicate Title,Image Quality,Image Match Category,AI Image Detected,Keyword Stuffing,Suspicious Word,Profanity,Adult Content,Illegal Product,Counterfeit,Category Mismatch,Title Description Mismatch,Spam Behavior,Risk Score,Risk Band,Confidence Score,Expected Result,Ground Truth,Reason,Business Rule,Acceptance Criteria,Created Date,Version,Remarks,__source_sheet__
0,DS000001,SC000001,DESCRIPTION_ANOMALY,LST-000001,SEL-01715,Sports,Dumbbell Set,Decathlon,Arcsaber 11,Decathlon Arcsaber 11,ขา่ย Decathlon Arcsaber 11 Dumbbell Set Like N...,"Actually this is a Modernform product, not Dec...",Poor,15497.86,15480,0.12,4.3,54,1,No,No,Medium,Yes,No,No,No,No,No,No,No,No,Yes,No,63,Medium Risk,0.86,REVIEW,MISLEADING_DESCRIPTION,Title advertises Decathlon Arcsaber 11 but des...,BR005,AC001,2026-05-14 00:04:04,1.0,Training record — auto-generated for training use,3_Training_Dataset
1,DS000002,SC000002,NORMAL,LST-000002,SEL-00250,Mobile Phone,Feature Phone,Xiaomi,Redmi Note 13,Xiaomi Redmi Note 13,ขาย Xiaomi Redmi Note 13 Feature Phone สภาพไม่...,ขาย Xiaomi Redmi Note 13 Feature Phone สภาพไม่...,Poor,6200.49,7140,-13.16,4.3,64,0,No,No,Medium,Yes,No,No,No,No,No,No,No,No,No,No,12,Normal,0.93,PASS,NORMAL,Listing appears consistent with normal marketp...,BR000,AC001,2025-10-26 05:20:38,1.0,Training record — auto-generated for training use,3_Training_Dataset
2,DS000003,SC000003,DUPLICATE,LST-000003,SEL-01765,Sports,Treadmill,Trek,Arcsaber 11,Trek Arcsaber 11,Trek Arcsaber 11 Treadmill - Fair condition fo...,Selling Trek Arcsaber 11 Treadmill. Condition:...,New,23289.68,23300,-0.04,3.3,24,0,Yes,Yes,High,Yes,No,No,No,No,No,No,No,No,No,No,70,Medium Risk,0.83,REVIEW,DUPLICATE_LISTING,Same seller re-posted an identical image under...,BR007,AC001,2025-07-06 05:28:31,1.0,Training record — auto-generated for training use,3_Training_Dataset


### 1.2 ตรวจสอบ null ที่ตั้งใจ (intentional) vs ที่ไม่ตั้งใจ

ตาม Data Dictionary ค่า null ใน `Product_Name`, `Condition`, `Price`, `Marketplace_Average_Price`,
`Price_Deviation_Percentage` ส่วนหนึ่งเป็น **NEGATIVE test case ที่จงใจใส่ไว้เพื่อทดสอบ** —
ห้ามลบทิ้งหรือ impute ทับโดยไม่เช็ค `Scenario_Type` ก่อน


In [14]:
null_cols = listing_all.columns[listing_all.isna().any()].tolist()
print("คอลัมน์ที่มี null:", null_cols)

for col in null_cols:
    null_mask = listing_all[col].isna()
    print(f"\n--- {col}: null ทั้งหมด {null_mask.sum()} แถว ---")
    print(listing_all.loc[null_mask, "Scenario Type"].value_counts())


คอลัมน์ที่มี null: ['Title', 'Price', 'Price Deviation Percentage']

--- Title: null ทั้งหมด 189 แถว ---
Scenario Type
NEGATIVE    189
Name: count, dtype: int64

--- Price: null ทั้งหมด 252 แถว ---
Scenario Type
NEGATIVE    252
Name: count, dtype: int64

--- Price Deviation Percentage: null ทั้งหมด 252 แถว ---
Scenario Type
NEGATIVE    252
Name: count, dtype: int64


In [15]:
# แยกธง is_intentional_null เก็บไว้ ไม่ลบแถว NEGATIVE ทิ้ง
# แต่ถ้า null เกิดขึ้นนอก scenario NEGATIVE/EDGE ให้ถือเป็น data quality issue จริง
INTENTIONAL_NULL_SCENARIOS = {"NEGATIVE", "EDGE"}

listing_all["is_intentional_null"] = listing_all["Scenario Type"].isin(INTENTIONAL_NULL_SCENARIOS)

unexpected_nulls = listing_all[
    listing_all[null_cols].isna().any(axis=1) & ~listing_all["is_intentional_null"]
]
print(f"แถวที่มี null แบบไม่ควรมี (นอก NEGATIVE/EDGE): {len(unexpected_nulls)}")
unexpected_nulls[["Dataset ID", "Scenario Type"] + null_cols].head(10)


แถวที่มี null แบบไม่ควรมี (นอก NEGATIVE/EDGE): 0


,Dataset ID,Scenario Type,Title,Price,Price Deviation Percentage


> ถ้าเจอ unexpected null (นอกกลุ่ม NEGATIVE/EDGE) ให้ตัดสินใจเป็นรายคอลัมน์ เช่น
> impute ด้วยค่ากลาง (median) สำหรับตัวเลข หรือ mode สำหรับ categorical — ในที่นี้จะ impute แบบระมัดระวัง

In [16]:
numeric_impute_cols = ["Price", "Marketplace Average Price", "Price Deviation Percentage"]
for col in numeric_impute_cols:
    if col in unexpected_nulls.columns and len(unexpected_nulls) > 0:
        median_val = listing_all.loc[~listing_all["is_intentional_null"], col].median()
        fix_mask = listing_all[col].isna() & ~listing_all["is_intentional_null"]
        listing_all.loc[fix_mask, col] = median_val

if "Product Name" in listing_all.columns:
    fix_mask = listing_all["Product Name"].isna() & ~listing_all["is_intentional_null"]
    listing_all.loc[fix_mask, "Product Name"] = (
        listing_all.loc[fix_mask, "Brand"].fillna("") + " " + listing_all.loc[fix_mask, "Model"].fillna("")
    ).str.strip()

print("เหลือ null นอก intentional case:",
      listing_all.loc[~listing_all["is_intentional_null"], null_cols].isna().sum().sum())


เหลือ null นอก intentional case: 0


### 1.3 ทำความสะอาดข้อความ (Title / Description)

ข้อความมีภาษาไทยปนอังกฤษ, emoji, ช่องว่างซ้ำ, ตัวอักษรพิเศษ — normalize ให้ใช้งานง่ายขึ้น
แต่ **เก็บ emoji/สัญลักษณ์ไว้เป็นฟีเจอร์แยก** ก่อนลบออกจากข้อความหลัก เพราะมันอาจสัมพันธ์กับ Spam/Suspicious_Word


In [17]:
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E0-\U0001F1FF"
    "]+", flags=re.UNICODE
)

def extract_and_clean_text(series):
    has_emoji = series.fillna("").str.contains(EMOJI_PATTERN)
    cleaned = series.fillna("").apply(lambda s: EMOJI_PATTERN.sub("", s))
    cleaned = cleaned.str.replace(r"\s+", " ", regex=True).str.strip()
    return cleaned, has_emoji

listing_all["Title_clean"], listing_all["Title_has_emoji"] = extract_and_clean_text(listing_all["Title"])
listing_all["Description_clean"], listing_all["Description_has_emoji"] = extract_and_clean_text(listing_all["Description"])

listing_all["Title_length"] = listing_all["Title_clean"].str.len()
listing_all["Description_length"] = listing_all["Description_clean"].str.len()

listing_all[["Title", "Title_clean", "Title_has_emoji", "Title_length"]].head(5)


,Title,Title_clean,Title_has_emoji,Title_length
0,ขา่ย Decathlon Arcsaber 11 Dumbbell Set Like N...,ขา่ย Decathlon Arcsaber 11 Dumbbell Set Like N...,False,57
1,ขาย Xiaomi Redmi Note 13 Feature Phone สภาพไม่...,ขาย Xiaomi Redmi Note 13 Feature Phone สภาพไม่...,True,63
2,Trek Arcsaber 11 Treadmill - Fair condition fo...,Trek Arcsaber 11 Treadmill - Fair condition fo...,True,54
3,ขาย Samsung MatePad 11 Tablet Like New ถูกๆ BNIB,ขาย Samsung MatePad 11 Tablet Like New ถูกๆ BNIB,False,48
4,ขาย Apple V29 Smartphone Like New ถูกๆ งานแท้,ขาย Apple V29 Smartphone Like New ถูกๆ งานแท้,False,45


### 1.4 Encode ฟีเจอร์ Yes/No → 0/1

คอลัมน์ flag เกือบทั้งหมดเป็น string 'Yes'/'No' — encode เป็นตัวเลขให้พร้อมเข้าโมเดล


In [18]:
yesno_cols = [
    "Duplicate Image", "Duplicate Title", "Image Match Category", "AI Image Detected",
    "Keyword Stuffing", "Suspicious Word", "Profanity", "Adult Content",
    "Illegal Product", "Counterfeit", "Category Mismatch",
    "Title Description Mismatch", "Spam Behavior",
]
yesno_cols = [c for c in yesno_cols if c in listing_all.columns]

for col in yesno_cols:
    listing_all[col + "_flag"] = listing_all[col].map({"Yes": 1, "No": 0})

listing_all[yesno_cols[:3] + [c + "_flag" for c in yesno_cols[:3]]].head(5)


,Duplicate Image,Duplicate Title,Image Match Category,Duplicate Image_flag,Duplicate Title_flag,Image Match Category_flag
0,No,No,Yes,0,0,1
1,No,No,Yes,0,0,1
2,Yes,Yes,Yes,1,1,1
3,No,No,Yes,0,0,1
4,No,No,Yes,0,0,1


### 1.5 ตรวจสอบขอบเขตค่า (range validation) ตาม Data Dictionary

- `Seller_Rating`: 0.0–5.0
- `Risk_Score`: 0–100
- `Confidence_Score`: 0.00–1.00
- `Price`, `Marketplace_Average_Price`: >= 0


In [19]:
range_checks = {
    "Seller Rating": (0.0, 5.0),
    "Risk Score": (0, 100),
    "Confidence Score": (0.0, 1.0),
}

for col, (lo, hi) in range_checks.items():
    out_of_range = ~listing_all[col].between(lo, hi)
    print(f"{col}: {out_of_range.sum()} แถวหลุดขอบเขต [{lo}, {hi}]")

# Price ต้อง >= 0 (ยกเว้น null ที่เป็น intentional)
neg_price = (listing_all["Price"] < 0)
print("Price ติดลบ:", neg_price.sum())


Seller Rating: 0 แถวหลุดขอบเขต [0.0, 5.0]
Risk Score: 0 แถวหลุดขอบเขต [0, 100]
Confidence Score: 0 แถวหลุดขอบเขต [0.0, 1.0]
Price ติดลบ: 0


### 1.6 Encode categorical columns สำหรับโมเดล

- Label encode / one-hot สำหรับ `Category`, `Condition`, `Image Quality`
- เก็บ label เป้าหมาย `Ground Truth` และ `Expected Result` แยกไว้ไม่ปนกับฟีเจอร์


In [20]:
categorical_cols = ["Category", "Subcategory", "Condition", "Image Quality", "Risk Band"]
categorical_cols = [c for c in categorical_cols if c in listing_all.columns]

# เก็บชื่อคอลัมน์ก่อน encode ไว้ เพื่อคำนวณว่าคอลัมน์ dummy ใหม่ๆ มีอะไรบ้าง
# (แม่นยำกว่าการเดาด้วย substring match ซึ่งอาจชนกับคอลัมน์อื่น เช่น 'Category_Mismatch')
cols_before = set(listing_all.columns)
listing_encoded = pd.get_dummies(listing_all, columns=categorical_cols, prefix=categorical_cols)
dummy_cols_listing = [c for c in listing_encoded.columns if c not in cols_before]

print("Shape หลัง one-hot encode:", listing_encoded.shape)
print(f"จำนวนคอลัมน์ dummy ที่สร้างใหม่: {len(dummy_cols_listing)}")


Shape หลัง one-hot encode: (25000, 181)
จำนวนคอลัมน์ dummy ที่สร้างใหม่: 121


### 1.7 แยกฟีเจอร์ (X) และ label (y) พร้อมแยกกลับเป็น Train/Val/Test/UAT ตามเดิม

In [21]:
FEATURE_COLS = (
    [c for c in listing_encoded.columns if c.endswith("_flag")]
    + dummy_cols_listing
    + ["Price", "Marketplace Average Price", "Price Deviation Percentage",
       "Seller Rating", "Seller Total Listings", "Seller Violation Count",
       "Title_length", "Description_length", "Title_has_emoji", "Description_has_emoji"]
)
FEATURE_COLS = [c for c in FEATURE_COLS if c in listing_encoded.columns]

LABEL_COLS = ["Ground Truth", "Expected Result", "Risk Score", "Risk Band"]

print(f"จำนวนฟีเจอร์: {len(FEATURE_COLS)}")

cleaned_splits = {}
for sheet, name in [("3_Training_Dataset", "train"), ("4_Validation_Dataset", "val"),
                     ("5_Testing_Dataset", "test"), ("6_UAT_Dataset", "uat")]:
    subset = listing_encoded[listing_encoded["__source_sheet__"] == sheet].copy()
    cleaned_splits[name] = subset
    out_path = os.path.join(OUTPUT_DIR, f"listing_{name}_clean.csv")
    subset.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"{name}: {subset.shape} -> {out_path}")


จำนวนฟีเจอร์: 144
train: (20000, 181) -> output/listing_train_clean.csv
val: (2000, 181) -> output/listing_val_clean.csv
test: (2000, 181) -> output/listing_test_clean.csv
uat: (1000, 181) -> output/listing_uat_clean.csv


### 1.8 ตัวอย่างเตรียม X, y พร้อมเข้าโมเดล classification

(ตัวอย่างสั้นๆ ใช้ RandomForest ทำนาย `Ground Truth` — ปรับ/เปลี่ยนโมเดลได้ตามต้องการ)


In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

X_train = cleaned_splits["train"][FEATURE_COLS].fillna(0)
X_val   = cleaned_splits["val"][FEATURE_COLS].fillna(0)

le = LabelEncoder()
y_train = le.fit_transform(cleaned_splits["train"]["Ground Truth"])
y_val   = le.transform(cleaned_splits["val"]["Ground Truth"])

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_val)
print(classification_report(y_val, y_pred, target_names=le.classes_, zero_division=0))


                        precision    recall  f1-score   support

              BOUNDARY       0.92      1.00      0.96       145
     CATEGORY_MISMATCH       1.00      1.00      1.00        73
           COUNTERFEIT       1.00      1.00      1.00        36
     DUPLICATE_LISTING       1.00      1.00      1.00        55
             EDGE_CASE       0.95      0.98      0.96        91
                 FRAUD       1.00      1.00      1.00        55
             HIGH_RISK       1.00      1.00      1.00        36
         INVALID_INPUT       0.99      0.98      0.98        91
          KEYWORD_SPAM       1.00      1.00      1.00        55
   LOW_QUALITY_LISTING       1.00      1.00      1.00        36
MISLEADING_DESCRIPTION       1.00      1.00      1.00       109
                NORMAL       1.00      1.00      1.00       818
         PRICE_ANOMALY       0.99      1.00      1.00       145
    PROHIBITED_PRODUCT       1.00      1.00      1.00        36
                  SCAM       1.00      

---
# PART 2 — Product Recommendation Dataset

โครงสร้างไฟล์: `01_Data_Dictionary`, `02_Business_Rules`, `03_Buyer_Profile`, `04_Buyer_Behavior`,
`05_Recommendation_Dataset`, `06_Cold_Start_Dataset`, `07_Recommendation_Feedback`,
`08_Coverage_Matrix`, `09_Acceptance_Criteria`

หมายเหตุ: ชื่อไฟล์ต้นทางระบุว่า "แนะนำคำอธิบายสินค้า" แต่เนื้อหาจริงคือระบบ **แนะนำสินค้า (recommendation)**
ไม่ใช่การ generate คำอธิบายสินค้า — โปรดตรวจสอบว่าตรงกับ use case ที่ต้องการ


In [23]:
xl2 = pd.ExcelFile(FILE_2)
print(xl2.sheet_names)


['01_Data_Dictionary', '02_Business_Rules', '03_Buyer_Profile', '04_Buyer_Behavior', '05_Recommendation_Dataset', '06_Cold_Start_Dataset', '07_Recommendation_Feedback', '08_Coverage_Matrix', '09_Acceptance_Criteria']


In [24]:
bus_rules_2 = xl2.parse("02_Business_Rules", header=3)
data_dict_2 = xl2.parse("01_Data_Dictionary", header=3)
bus_rules_2


,Rule_ID,Module,Rule_Description,Priority,Acceptance_Criteria
0,BR001,Recommendation,Recommend products based on buyer behavior (se...,High,AC001
1,BR002,Recommendation,Never recommend sold products.,Critical,AC003
2,BR003,Recommendation,Never recommend deleted products.,Critical,AC003
3,BR004,Recommendation,Never recommend inactive products.,Critical,AC003
4,BR005,Recommendation,"Higher priority for Recently Viewed, Wishlist,...",High,AC001
5,BR006,Recommendation,"If no user history exists, use Popular / Trend...",High,AC006
6,BR007,Recommendation,"Recommendations must consider Category, Brand,...",High,AC004
7,BR008,Privacy,"Never expose Phone Number, Email, Address, Ban...",Critical,AC007
8,BR009,Data Quality,No duplicated Buyer_ID + Product_ID combinatio...,High,AC008


In [25]:
buyer_profile  = xl2.parse("03_Buyer_Profile", header=3)
buyer_behavior = xl2.parse("04_Buyer_Behavior", header=3)
reco           = xl2.parse("05_Recommendation_Dataset", header=3)
cold_start     = xl2.parse("06_Cold_Start_Dataset", header=3)
feedback       = xl2.parse("07_Recommendation_Feedback", header=3)

for name, df in [("buyer_profile", buyer_profile), ("buyer_behavior", buyer_behavior),
                  ("reco", reco), ("cold_start", cold_start), ("feedback", feedback)]:
    print(name, df.shape)


buyer_profile (1000, 16)
buyer_behavior (5000, 20)
reco (10000, 32)
cold_start (500, 17)
feedback (5000, 15)


### 2.1 บังคับ Business Rules ก่อนทำความสะอาดอื่น (BR002–BR004, BR009)

- **BR002-BR004**: ห้ามแนะนำสินค้าที่ `Product_Status` เป็น Sold / Inactive / Deleted
- **BR009**: ห้ามมี `Buyer_ID` + `Product_ID` ซ้ำกันใน Recommendation Dataset


In [26]:
print("Product_Status ก่อนกรอง:")
print(reco["Product_Status"].value_counts())

# BR002-BR004: กรองเฉพาะ Active เป็นชุดเทรนหลัก แต่เก็บชุดที่ไม่ Active ไว้ตรวจสอบว่าถูก label REJECT ถูกต้องหรือไม่
invalid_status_but_accepted = reco[
    (reco["Product_Status"] != "Active") & (reco["Expected_Result"].str.startswith("Accept"))
]
print(f"\nพบ {len(invalid_status_but_accepted)} แถวที่ Product_Status ไม่ Active แต่ label เป็น Accept -> ผิด Business Rule ต้องตรวจสอบ/แก้ label")
invalid_status_but_accepted[["Dataset_ID", "Buyer_ID", "Product_ID", "Product_Status", "Expected_Result"]].head(10)


Product_Status ก่อนกรอง:
Product_Status
Active      9063
Sold         538
Inactive     265
Deleted      134
Name: count, dtype: int64

พบ 0 แถวที่ Product_Status ไม่ Active แต่ label เป็น Accept -> ผิด Business Rule ต้องตรวจสอบ/แก้ label


,Dataset_ID,Buyer_ID,Product_ID,Product_Status,Expected_Result


In [27]:
# BR009: เช็ค duplicate Buyer_ID + Product_ID
dup_mask = reco.duplicated(subset=["Buyer_ID", "Product_ID"], keep=False)
print(f"แถวที่ Buyer_ID+Product_ID ซ้ำ: {dup_mask.sum()}")

reco_dedup = reco.drop_duplicates(subset=["Buyer_ID", "Product_ID"], keep="first").copy()
print("Shape ก่อน/หลัง dedup:", reco.shape, "->", reco_dedup.shape)


แถวที่ Buyer_ID+Product_ID ซ้ำ: 0
Shape ก่อน/หลัง dedup: (10000, 32) -> (10000, 32)


### 2.2 แยก pipe-delimited fields เป็น list

`Search_History`, `View_History`, `Click_History`, `Wishlist`, `Favorite`, `Purchase_History`
เก็บเป็นข้อความคั่นด้วย `|` — แปลงเป็น list เพื่อใช้งานง่ายขึ้น (เช่น เข้า embedding model)


In [28]:
def split_pipe(series):
    return series.fillna("").apply(
        lambda s: [x.strip() for x in s.split("|") if x.strip()] if isinstance(s, str) else []
    )

pipe_cols_behavior = ["Search_History", "View_History", "Click_History",
                       "Wishlist", "Favorite", "Purchase_History"]
pipe_cols_behavior = [c for c in pipe_cols_behavior if c in buyer_behavior.columns]

for col in pipe_cols_behavior:
    buyer_behavior[col + "_list"] = split_pipe(buyer_behavior[col])
    buyer_behavior[col + "_count"] = buyer_behavior[col + "_list"].apply(len)

buyer_behavior[[c for c in pipe_cols_behavior] + [c + "_count" for c in pipe_cols_behavior]].head(5)


,Search_History,View_History,Click_History,Wishlist,Favorite,Purchase_History,Search_History_count,View_History_count,Click_History_count,Wishlist_count,Favorite_count,Purchase_History_count
0,office chair ergonomic | เก้าอี้เกมมิ่ง | ipho...,PRD002310 | PRD001945,NaN,NaN,NaN,NaN,3,2,0,0,0,0
1,ตู้เย็น 2 ประตู 📦 | กล้องฟิล์ม | โน๊ตบุ๊คถูกๆ ...,PRD001909 | PRD001989 | PRD002634 | PRD000169 ...,PRD002634 | PRD000169 | PRD000666 | PRD000012 ...,PRD002634 | PRD000666 | PRD002666 | PRD002631,PRD001821,PRD001840,27,39,23,4,1,1
2,โนีตบุค | กล้องฟิล์ม | refrigerator 2 door | o...,PRD001676 | PRD002649 | PRD000916 | PRD001987 ...,PRD001676 | PRD000916 | PRD001964 | PRD002851 ...,PRD002349,PRD001987 | PRD001964,PRD001676 | PRD002349 | PRD000700,12,10,6,1,2,3
3,รองเท้าไซส์ 40 | กระเป๋าแบรนด์เนม 🥳 | labtop ส...,PRD000036 | PRD001217 | PRD002835 | PRD001121 ...,PRD001587 | PRD000198 | PRD000974 | PRD001872 ...,PRD002440,NaN,PRD001587,3,68,6,1,0,1
4,NaN,PRD001512 | PRD000417,NaN,NaN,NaN,NaN,0,2,0,0,0,0


In [29]:
# ทำแบบเดียวกันกับ cold_start dataset
pipe_cols_cold = ["Search_History", "View_History", "Wishlist", "Favorite",
                   "Purchase_History", "Recommended_Products"]
pipe_cols_cold = [c for c in pipe_cols_cold if c in cold_start.columns]

for col in pipe_cols_cold:
    cold_start[col + "_list"] = split_pipe(cold_start[col])
    cold_start[col + "_count"] = cold_start[col + "_list"].apply(len)

cold_start[pipe_cols_cold[:2] + [c + "_count" for c in pipe_cols_cold[:2]]].head(5)


,Search_History,View_History,Search_History_count,View_History_count
0,NaN,NaN,0,0
1,NaN,NaN,0,0
2,NaN,NaN,0,0
3,NaN,NaN,0,0
4,NaN,NaN,0,0


### 2.3 ตรวจสอบข้อมูลอ่อนไหว (Privacy — BR008)

BR008 ห้ามมีข้อมูล PDPA เช่น เบอร์โทร/อีเมล/บัตรประชาชน/พิกัด GPS ปนอยู่ในฟิลด์ข้อความอิสระ
(`Search_History`, `Feedback`) — สแกนคร่าวๆ ด้วย regex ก่อนใช้งานจริง


In [30]:
PII_PATTERNS = {
    "phone": r"0[689]\d{8}|0\d{1,2}-?\d{3}-?\d{4}",
    "email": r"[\w.+-]+@[\w-]+\.[\w.-]+",
    "citizen_id": r"\b\d{13}\b",
}

def scan_pii(series, col_name):
    hits = {}
    for label, pattern in PII_PATTERNS.items():
        mask = series.fillna("").astype(str).str.contains(pattern, regex=True)
        if mask.sum() > 0:
            hits[label] = mask.sum()
    return hits

for col in ["Search_History"]:
    if col in buyer_behavior.columns:
        print(col, "->", scan_pii(buyer_behavior[col], col))

if "Feedback" in feedback.columns:
    print("Feedback ->", scan_pii(feedback["Feedback"], "Feedback"))


Search_History -> {}
Feedback -> {}


### 2.4 ตรวจสอบขอบเขตค่า (0.00–1.00 scores) และความสอดคล้องของตัวเลข

- `Popularity_Score`, `Trending_Score`, `Embedding_Similarity`, `Category_Similarity`,
  `Brand_Similarity`, `Price_Similarity`, `Recommendation_Score`: ต้องอยู่ 0.00–1.00
- `Confidence_Score`: ต้องอยู่ 0.50–1.00
- `Historical_Purchases <= Historical_Clicks <= Historical_Views` (ตาม Data Dictionary)


In [31]:
score_cols_0_1 = ["Popularity_Score", "Trending_Score", "Embedding_Similarity",
                   "Category_Similarity", "Brand_Similarity", "Price_Similarity",
                   "Recommendation_Score"]
score_cols_0_1 = [c for c in score_cols_0_1 if c in reco_dedup.columns]

for col in score_cols_0_1:
    out_of_range = ~reco_dedup[col].between(0.0, 1.0)
    print(f"{col}: {out_of_range.sum()} แถวหลุดขอบเขต [0,1]")

if "Confidence_Score" in reco_dedup.columns:
    bad_conf = ~reco_dedup["Confidence_Score"].between(0.5, 1.0)
    print(f"Confidence_Score: {bad_conf.sum()} แถวหลุดขอบเขต [0.5,1.0]")

# ความสอดคล้อง Historical_*
inconsistent = reco_dedup[
    (reco_dedup["Historical_Purchases"] > reco_dedup["Historical_Clicks"]) |
    (reco_dedup["Historical_Clicks"] > reco_dedup["Historical_Views"])
]
print(f"\nแถวที่ Historical_Purchases/Clicks/Views ไม่สอดคล้องกัน: {len(inconsistent)}")


Popularity_Score: 0 แถวหลุดขอบเขต [0,1]
Trending_Score: 0 แถวหลุดขอบเขต [0,1]
Embedding_Similarity: 0 แถวหลุดขอบเขต [0,1]
Category_Similarity: 0 แถวหลุดขอบเขต [0,1]
Brand_Similarity: 0 แถวหลุดขอบเขต [0,1]
Price_Similarity: 0 แถวหลุดขอบเขต [0,1]
Recommendation_Score: 0 แถวหลุดขอบเขต [0,1]
Confidence_Score: 0 แถวหลุดขอบเขต [0.5,1.0]

แถวที่ Historical_Purchases/Clicks/Views ไม่สอดคล้องกัน: 0


### 2.5 Join ตารางเข้าด้วยกันให้เป็น feature table เดียว

รวม `Recommendation_Dataset` (แกนหลัก) กับ `Buyer_Profile` ผ่าน `Buyer_ID`


In [32]:
reco_joined = reco_dedup.merge(
    buyer_profile,
    on="Buyer_ID",
    how="left",
    suffixes=("", "_buyer_profile")
)
print("Shape หลัง join กับ Buyer_Profile:", reco_joined.shape)

missing_profile = reco_joined["Age_Group"].isna().sum()
print(f"แถวที่ Buyer_ID ไม่มีใน Buyer_Profile (อาจเป็น cold-start buyer): {missing_profile}")


Shape หลัง join กับ Buyer_Profile: (10000, 47)
แถวที่ Buyer_ID ไม่มีใน Buyer_Profile (อาจเป็น cold-start buyer): 0


### 2.6 Encode categorical + เตรียม X, y สำหรับ ranking/classification model

In [33]:
# หมายเหตุ: ตั้งใจไม่ใส่ 'Preferred_Category' / 'Preferred_Brand' ใน one-hot ตรงนี้
# เพราะ cardinality สูงและซ้ำซ้อนกับ Category_Similarity/Brand_Similarity ที่คำนวณไว้แล้ว
cat_cols_reco = ["Category", "Subcategory", "Brand", "Condition",
                  "Member_Level", "Activity_Level", "Age_Group"]
cat_cols_reco = [c for c in cat_cols_reco if c in reco_joined.columns]

# เก็บชื่อคอลัมน์ก่อน encode ไว้ เพื่อคำนวณว่าคอลัมน์ dummy ใหม่ๆ มีอะไรบ้าง
# (แม่นยำกว่าการเดาด้วย substring match ซึ่งอาจชนกับคอลัมน์อื่น เช่น 'Category_Similarity', 'Preferred_Brand')
cols_before_reco = set(reco_joined.columns)
reco_encoded = pd.get_dummies(reco_joined, columns=cat_cols_reco, prefix=cat_cols_reco)
dummy_cols_reco = [c for c in reco_encoded.columns if c not in cols_before_reco]

FEATURE_COLS_RECO = (
    score_cols_0_1
    + ["Historical_Clicks", "Historical_Views", "Historical_Purchases",
       "Seller_Rating", "Price"]
    + dummy_cols_reco
)
FEATURE_COLS_RECO = [c for c in FEATURE_COLS_RECO if c in reco_encoded.columns]

print(f"จำนวนฟีเจอร์: {len(FEATURE_COLS_RECO)}")

out_path = os.path.join(OUTPUT_DIR, "recommendation_clean.csv")
reco_encoded.to_csv(out_path, index=False, encoding="utf-8-sig")
print("บันทึกแล้ว ->", out_path)

cold_start.to_csv(os.path.join(OUTPUT_DIR, "cold_start_clean.csv"), index=False, encoding="utf-8-sig")
buyer_behavior.to_csv(os.path.join(OUTPUT_DIR, "buyer_behavior_clean.csv"), index=False, encoding="utf-8-sig")
feedback.to_csv(os.path.join(OUTPUT_DIR, "feedback_clean.csv"), index=False, encoding="utf-8-sig")
print("บันทึก cold_start / buyer_behavior / feedback เรียบร้อย")


จำนวนฟีเจอร์: 212
บันทึกแล้ว -> output/recommendation_clean.csv
บันทึก cold_start / buyer_behavior / feedback เรียบร้อย


### 2.7 ตัวอย่างเตรียมโมเดล ranking/classification

ทำนาย `Expected_Result` (Accept/Reject/Accept-Popularity Based) จากฟีเจอร์ similarity + historical


In [34]:
from sklearn.model_selection import train_test_split

X = reco_encoded[FEATURE_COLS_RECO].fillna(0)
le2 = LabelEncoder()
y = le2.fit_transform(reco_encoded["Expected_Result"])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf2 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf2.fit(X_tr, y_tr)

y_pred2 = clf2.predict(X_te)
print(classification_report(y_te, y_pred2, target_names=le2.classes_, zero_division=0))


                           precision    recall  f1-score   support

                   Accept       0.94      1.00      0.97      1709
Accept - Popularity Based       0.00      0.00      0.00       104
                   Reject       0.97      0.99      0.98       187

                 accuracy                           0.94      2000
                macro avg       0.64      0.66      0.65      2000
             weighted avg       0.90      0.94      0.92      2000



---
## สรุปไฟล์ output ที่ได้

| ไฟล์ | เนื้อหา |
|---|---|
| `output/listing_train_clean.csv` / `_val_` / `_test_` / `_uat_` | Abnormal Listing Detection แต่ละชุด (ทำความสะอาด + encode แล้ว) |
| `output/recommendation_clean.csv` | Recommendation Dataset หลัก (join กับ Buyer Profile + encode แล้ว) |
| `output/cold_start_clean.csv` | Cold-start dataset (แยก list แล้ว) |
| `output/buyer_behavior_clean.csv` | Buyer Behavior (แยก pipe-delimited fields แล้ว) |
| `output/feedback_clean.csv` | Recommendation Feedback |

ขั้นตอนถัดไปที่แนะนำ: ปรับ hyperparameter, ลอง gradient boosting (XGBoost/LightGBM),
และเพิ่ม cross-validation แทนการแบ่ง train/test ครั้งเดียว
